# Lab 2.2: Machine Learning Fundamentals

## Learning Objectives
- Apply linear regression for prediction
- Use logistic regression for classification
- Build interpretable decision trees
- Compare model performance

### Setup

In [ ]:
# Essential imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import mean_squared_error, accuracy_score

# Setup
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
print("Ready for machine learning!")

## 1. Linear Regression - Predicting Numbers (35 min)

**What**: Find the best line through data points to predict continuous values
**When**: Predicting numbers (prices, scores, ratings)
**Example**: Predict philosopher influence from publication count

In [ ]:
# Create philosopher influence dataset
data = {
    'publications': [5, 10, 15, 20, 25, 30, 8, 12, 18, 22],
    'citations': [50, 120, 200, 300, 400, 500, 80, 150, 250, 350],
    'influence_score': [6.2, 7.1, 7.8, 8.5, 8.9, 9.2, 6.8, 7.3, 8.1, 8.7]
}

df = pd.DataFrame(data)
print("Philosopher Dataset:")
print(df.head())

# Quick visualization
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(df['publications'], df['influence_score'])
plt.xlabel('Publications')
plt.ylabel('Influence Score')
plt.title('Publications vs Influence')

plt.subplot(1, 2, 2)
plt.scatter(df['citations'], df['influence_score'])
plt.xlabel('Citations')
plt.ylabel('Influence Score')
plt.title('Citations vs Influence')
plt.tight_layout()
plt.show()

In [ ]:
# Train linear regression
X = df[['publications', 'citations']]
y = df['influence_score']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
print(f"Model Performance:")
print(f"Mean Squared Error: {mse:.3f}")
print(f"\nCoefficients:")
print(f"Publications: {model.coef_[0]:.4f}")
print(f"Citations: {model.coef_[1]:.4f}")
print(f"Intercept: {model.intercept_:.4f}")

# Interpretation
print(f"\nInterpretation:")
print(f"Each publication increases influence by {model.coef_[0]:.4f}")
print(f"Each citation increases influence by {model.coef_[1]:.4f}")

### Exercise 1: Student Performance Prediction
Predict test scores from study habits.

In [ ]:
# Student performance dataset
student_data = {
    'study_hours': [2, 4, 6, 3, 8, 5, 7, 4, 6, 5],
    'sleep_hours': [6, 7, 8, 6, 8, 7, 9, 6, 8, 7],
    'test_score': [72, 78, 85, 75, 92, 80, 88, 79, 86, 81]
}

student_df = pd.DataFrame(student_data)
print("Student Dataset:")
print(student_df)

# TODO: Your turn
# 1. Create X (features) and y (target)
X_student = student_df[['study_hours', 'sleep_hours']]
y_student = student_df['test_score']

# 2. Split data
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_student, y_student, test_size=0.3, random_state=42
)

# 3. Train model
student_model = LinearRegression()
student_model.fit(X_train_s, y_train_s)

# 4. Evaluate
y_pred_s = student_model.predict(X_test_s)
mse_s = mean_squared_error(y_test_s, y_pred_s)

print(f"\nStudent Model Results:")
print(f"MSE: {mse_s:.2f}")
print(f"Study hours coefficient: {student_model.coef_[0]:.2f}")
print(f"Sleep hours coefficient: {student_model.coef_[1]:.2f}")

## 2. Logistic Regression - Predicting Categories (25 min)

**What**: Predict categories/classes using probability
**When**: Yes/No, Type A/B, Category prediction
**Example**: Predict learning preference (Online/In-Person)

In [ ]:
# Learning preference dataset
np.random.seed(42)
n_samples = 30
ages = np.random.randint(18, 65, n_samples)
tech_comfort = np.random.randint(1, 11, n_samples)

# Create realistic preferences
preferences = []
for age, tech in zip(ages, tech_comfort):
    score = (65 - age) * 0.02 + tech * 0.08 + np.random.random() * 0.3
    preferences.append('Online' if score > 0.5 else 'In-Person')

pref_df = pd.DataFrame({
    'age': ages,
    'tech_comfort': tech_comfort,
    'preference': preferences
})

print(f"Learning Preference Dataset ({len(pref_df)} samples):")
print(pref_df.head())
print(f"\nPreference counts:")
print(pref_df['preference'].value_counts())

In [ ]:
# Train logistic regression
X_log = pref_df[['age', 'tech_comfort']]
y_log = pref_df['preference']

# Split data
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_log, y_log, test_size=0.3, random_state=42
)

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_log)
X_test_scaled = scaler.transform(X_test_log)

# Train model
log_model = LogisticRegression(random_state=42)
log_model.fit(X_train_scaled, y_train_log)

# Make predictions
y_pred_log = log_model.predict(X_test_scaled)
y_prob = log_model.predict_proba(X_test_scaled)

# Evaluate
accuracy = accuracy_score(y_test_log, y_pred_log)
print(f"Logistic Regression Results:")
print(f"Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")
print(f"\nSample predictions:")
for i in range(min(5, len(y_test_log))):
    print(f"Actual: {y_test_log.iloc[i]}, Predicted: {y_pred_log[i]}, Prob: {y_prob[i].max():.3f}")

### Exercise 2: Philosophy School Classification
Predict philosophical school from argument characteristics.

In [ ]:
# Philosophy school dataset
phil_data = {
    'logic_emphasis': [8, 9, 6, 7, 5, 8, 4, 9, 6, 7],  # 1-10 scale
    'empirical_focus': [3, 2, 8, 7, 9, 4, 8, 3, 7, 6],  # 1-10 scale
    'school': ['Analytic', 'Analytic', 'Continental', 'Continental', 'Pragmatic',
               'Analytic', 'Continental', 'Analytic', 'Continental', 'Pragmatic']
}

phil_df = pd.DataFrame(phil_data)
print("Philosophy School Dataset:")
print(phil_df)

# TODO: Your turn
# 1. Prepare data
X_phil = phil_df[['logic_emphasis', 'empirical_focus']]
y_phil = phil_df['school']

# 2. Split and scale
X_train_phil, X_test_phil, y_train_phil, y_test_phil = train_test_split(
    X_phil, y_phil, test_size=0.3, random_state=42
)

scaler_phil = StandardScaler()
X_train_phil_scaled = scaler_phil.fit_transform(X_train_phil)
X_test_phil_scaled = scaler_phil.transform(X_test_phil)

# 3. Train logistic regression
phil_model = LogisticRegression(random_state=42, max_iter=1000)
phil_model.fit(X_train_phil_scaled, y_train_phil)

# 4. Evaluate
y_pred_phil = phil_model.predict(X_test_phil_scaled)
accuracy_phil = accuracy_score(y_test_phil, y_pred_phil)

print(f"\nPhilosophy School Classification:")
print(f"Accuracy: {accuracy_phil:.3f}")
print(f"Classes: {phil_model.classes_}")

## 3. Decision Trees - Interpretable Decisions (30 min)

**What**: Make decisions using yes/no questions in a tree structure
**When**: Need interpretable models, rule-based decisions
**Example**: Predict research funding approval

In [ ]:
# Research funding dataset
funding_data = {
    'research_score': [8, 6, 9, 5, 7, 8, 4, 9, 6, 7, 8, 5],
    'budget_reasonable': [1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0],  # 1=Yes, 0=No
    'prior_grants': [2, 0, 3, 1, 2, 3, 0, 4, 1, 2, 3, 0],
    'approved': ['Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'No']
}

funding_df = pd.DataFrame(funding_data)
print("Research Funding Dataset:")
print(funding_df)
print(f"\nApproval rate: {(funding_df['approved'] == 'Yes').mean():.2%}")

In [ ]:
# Train decision tree
X_tree = funding_df[['research_score', 'budget_reasonable', 'prior_grants']]
y_tree = funding_df['approved']

# Split data
X_train_tree, X_test_tree, y_train_tree, y_test_tree = train_test_split(
    X_tree, y_tree, test_size=0.3, random_state=42
)

# Train tree (no scaling needed!)
tree_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)
tree_model.fit(X_train_tree, y_train_tree)

# Make predictions
y_pred_tree = tree_model.predict(X_test_tree)
accuracy_tree = accuracy_score(y_test_tree, y_pred_tree)

print(f"Decision Tree Results:")
print(f"Accuracy: {accuracy_tree:.3f}")
print(f"Tree depth: {tree_model.get_depth()}")
print(f"Number of leaves: {tree_model.get_n_leaves()}")

# Feature importance
feature_names = ['Research Score', 'Budget Reasonable', 'Prior Grants']
importances = tree_model.feature_importances_
print(f"\nFeature Importance:")
for feature, importance in zip(feature_names, importances):
    print(f"{feature}: {importance:.3f}")

In [ ]:
# Visualize the decision tree
plt.figure(figsize=(12, 8))
plot_tree(tree_model,
          feature_names=feature_names,
          class_names=['No', 'Yes'],
          filled=True,
          fontsize=10,
          rounded=True)
plt.title('Research Funding Decision Tree')
plt.show()

print("Reading the tree:")
print("- Each box shows a decision rule")
print("- Follow paths from top to bottom")
print("- Orange = 'No', Blue = 'Yes'")
print("- Darker colors = more confident predictions")

### Exercise 3: Journal Acceptance Prediction
Build a decision tree to predict journal acceptance.

In [ ]:
# Journal acceptance dataset
journal_data = {
    'novelty_score': [8, 6, 9, 5, 7, 8, 4, 9, 6, 7],
    'methodology_score': [7, 8, 8, 6, 9, 7, 5, 9, 7, 8],
    'clarity_score': [6, 7, 9, 5, 8, 6, 4, 8, 6, 7],
    'accepted': ['Yes', 'Yes', 'Yes', 'No', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes']
}

journal_df = pd.DataFrame(journal_data)
print("Journal Acceptance Dataset:")
print(journal_df)

# TODO: Your turn
# 1. Prepare data
X_journal = journal_df[['novelty_score', 'methodology_score', 'clarity_score']]
y_journal = journal_df['accepted']

# 2. Train decision tree
journal_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
journal_tree.fit(X_journal, y_journal)

# 3. Evaluate
y_pred_journal = journal_tree.predict(X_journal)
accuracy_journal = accuracy_score(y_journal, y_pred_journal)

print(f"\nJournal Acceptance Model:")
print(f"Accuracy: {accuracy_journal:.3f}")

# 4. Show feature importance
features = ['Novelty', 'Methodology', 'Clarity']
importances_journal = journal_tree.feature_importances_
print(f"\nMost important factor:")
most_important = features[np.argmax(importances_journal)]
print(f"{most_important} (importance: {max(importances_journal):.3f})")

## 4. Model Comparison & Selection (10 min)

**Key Question**: Which algorithm is best for your problem?

In [ ]:
# Compare models on the same classification task
# Using the learning preference dataset

print("=== MODEL COMPARISON ===")
print("Task: Predict learning preference (Online/In-Person)")
print("Features: Age, Tech Comfort")

# Prepare data
X_comp = pref_df[['age', 'tech_comfort']]
y_comp = pref_df['preference']
X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
    X_comp, y_comp, test_size=0.3, random_state=42
)

# Scale for logistic regression
scaler_comp = StandardScaler()
X_train_scaled_comp = scaler_comp.fit_transform(X_train_comp)
X_test_scaled_comp = scaler_comp.transform(X_test_comp)

# Train both models
log_comp = LogisticRegression(random_state=42)
tree_comp = DecisionTreeClassifier(max_depth=3, random_state=42)

log_comp.fit(X_train_scaled_comp, y_train_comp)
tree_comp.fit(X_train_comp, y_train_comp)

# Compare accuracy
log_accuracy = accuracy_score(y_test_comp, log_comp.predict(X_test_scaled_comp))
tree_accuracy = accuracy_score(y_test_comp, tree_comp.predict(X_test_comp))

print(f"\nResults:")
print(f"Logistic Regression: {log_accuracy:.3f}")
print(f"Decision Tree: {tree_accuracy:.3f}")

# Choose best model
if log_accuracy > tree_accuracy:
    print(f"\nWinner: Logistic Regression")
    print(f"Advantage: Better accuracy ({log_accuracy:.3f} vs {tree_accuracy:.3f})")
elif tree_accuracy > log_accuracy:
    print(f"\nWinner: Decision Tree")
    print(f"Advantage: Better accuracy ({tree_accuracy:.3f} vs {log_accuracy:.3f})")
else:
    print(f"\nTie! Both models perform equally well.")
    print(f"Choose based on interpretability needs.")

print(f"\nDecision Guide:")
print(f"- Choose Decision Tree if you need explainable rules")
print(f"- Choose Logistic Regression if you need probability estimates")
print(f"- Choose Linear Regression if predicting continuous numbers")

## Summary & Next Steps

### What You've Learned ✓
- **Linear Regression**: Predict numbers from features
- **Logistic Regression**: Predict categories with probabilities
- **Decision Trees**: Make interpretable rule-based decisions
- **Model Evaluation**: Compare performance using metrics

### Algorithm Selection Guide
| Task | Algorithm | Why |
|------|-----------|-----|
| Predict numbers | Linear Regression | Simple, interpretable |
| Predict categories | Logistic Regression | Gives probabilities |
| Need rules/explanation | Decision Tree | Human-readable decisions |

### Key Takeaways
1. **Always visualize your data first**
2. **Scale features for logistic regression**
3. **Decision trees don't need scaling**
4. **Choose algorithm based on your specific needs**

### Coming Next Week
- More advanced algorithms
- Real philosophical datasets
- Bias detection and fairness
- Handling text data for philosophy research

**Practice Tip**: Try these algorithms on your own research data!